# Introduction to Neural Network Architectures

- [View Solution Notebook](./solutions.html)
- [View Project Page](https://www.codecademy.com/content-items/c3e07b9eb8e5444b8b466ab95732c3c0)
  
In this project, you apply your knowledge of neural network architectures by building and training various models for two tasks using popular real-world datasets:

## Objectives

**A.** Build and compare a simple MLP and a deep CNN on the classic MNIST handwritten digit classification problem.
- Load and preprocess the [MNIST dataset](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html)
- Build an MLP architecture
- Build a ResNet18 architecture
- Profile, evaluate, and compare their performance

**B.** Build and compare an RNN, GRU, and LSTM to predict hourly household power usage from historical measurements.
- Load the [Household Electric Power Consumption dataset](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption)
- Construct and preprocess sequences for model input
- Build and train RNN, GRU, and LSTM models
- Profile, evaluate, and compare their performance

### Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as M
from sklearn.metrics import accuracy_score, classification_report
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Objective A: Predict Handwritten Images

## Task Group 1: MNIST Data Loading and Preprocessing

**Introduction to the MNIST Dataset**

The [MNIST dataset](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html) is one of the most popular and widely used benchmark datasets for image classification in machine learning. It consists of **70,000 images** of handwritten digits (0–9), each sized **28 × 28 pixels**, with a single grayscale channel. By default, 60,000 images are used for training, and 10,000 images are used for testing.

MNIST is commonly used for:
- Practicing image preprocessing, model training, and evaluation workflows.
- Building and evaluating image classification models.  
- Benchmarking neural network architectures such as MLPs and CNNs.  

Below is one sample image from each class (0–9):

![MNIST sample digits](mnist_samples.jpg)

### Task 1: Define Image Transformations
Define transformations to:
- Resize images to 28x28 (same as original)
- Convert to tensors
- Normalize the single channel, respectively, with mean `[0.1307]` and standard deviation `[0.3081]`

In [2]:
## YOUR SOLUTION HERE ##
transform = transforms.Compose([
    transforms.Resize(28),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307],
                         std=[0.3081])
])

### Task 2: Load MNIST Datasets
Load both the MNIST training and test datasets with the transforms you defined into the `datasets` directory.

The MNIST datasets can be located using `datasets.MNIST` from the `torchvision.datasets` module.

In [3]:
## YOUR SOLUTION HERE ##
data_dir = './datasets'
train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)

### Task 3: Create DataLoaders
Create DataLoader objects with:
- Training: batch size of `32` and set `shuffle=True`.
- Testing: batch size of `32` and set `shuffle=False`.

In [4]:
## YOUR SOLUTION HERE ##
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Show output - Verify data loading
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 60000
Test samples: 10000


## Task Group 2: Multi-Layer Perceptron (MLP) and ResNet18 Architectures

### Task 4: Define SimpleMLP Class
Create a **simple MLP** with the following default values:
- Input size: 28x28x1 (flattened image with one channel)
- Hidden size: 128
- Output size: 10 (digit classes)
- 3 fully connected layers with ReLU activations after the first and second fully connected layers

In [5]:
## YOUR SOLUTION HERE ##
class SimpleMLP(nn.Module):
    def __init__(self, input_size=28*28*1, hidden_size=128, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

### Task 5: Instantiate MLP Model
Create an instance of **SimpleMLP** class and move it to the device.  Be sure to correctly specify the default input size, hidden size, and output size. 

Profile the model size using an input size with the format: `(batch_size, channels, width, height)`

In [6]:
## YOUR SOLUTION HERE ##
model_mlp = SimpleMLP(input_size=28*28*1, hidden_size=128, output_size=10)
model_mlp = model_mlp.to(device)

# Show output - Model Size
from custom_torchinfo import custom_summary
custom_summary(model_mlp, input_size=(32, 1, 28, 28))

        Layer (type)              Output Shape         Param #
           Flatten-1                 [32, 784]               0
            Linear-2                 [32, 128]         100,480
              ReLU-3                 [32, 128]               0
            Linear-4                 [32, 128]          16,512
              ReLU-5                 [32, 128]               0
            Linear-6                  [32, 10]           1,290
         SimpleMLP-7                  [32, 10]               0
Total params: 118,282
Trainable params: 118,282
Non-trainable params: 0


### Task 6: Load and Modify ResNet18
Load **ResNet18** without pretrained weights and modify the final layer for 10 classes. Be sure to place the model to the device.

We'll need to modify the convolutional layer to accept a single channel (from three) using:

```py
model_resnet18.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_resnet18.maxpool = nn.Identity() # removes maxpool layer
```

Profile the model size using an input size with the format: `(batch_size, channels, width, height)`

In [7]:
## YOUR SOLUTION HERE ##
model_resnet18 = M.resnet18(weights=None)
model_resnet18.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_resnet18.maxpool = nn.Identity()

model_resnet18.fc = nn.Linear(model_resnet18.fc.in_features, 10)
model_resnet18 = model_resnet18.to(device)

# Show output - Display model architecture
from custom_torchinfo import custom_summary
custom_summary(model_resnet18, input_size=(32, 1, 28, 28))

        Layer (type)              Output Shape         Param #
            Conv2d-1          [32, 64, 28, 28]             576
       BatchNorm2d-2          [32, 64, 28, 28]             128
              ReLU-3          [32, 64, 28, 28]               0
          Identity-4          [32, 64, 28, 28]               0
            Conv2d-5          [32, 64, 28, 28]          36,864
       BatchNorm2d-6          [32, 64, 28, 28]             128
              ReLU-7          [32, 64, 28, 28]               0
            Conv2d-8          [32, 64, 28, 28]          36,864
       BatchNorm2d-9          [32, 64, 28, 28]             128
             ReLU-10          [32, 64, 28, 28]               0
       BasicBlock-11          [32, 64, 28, 28]               0
           Conv2d-12          [32, 64, 28, 28]          36,864
      BatchNorm2d-13          [32, 64, 28, 28]             128
             ReLU-14          [32, 64, 28, 28]               0
           Conv2d-15          [32, 64, 28, 28]         

## Task Group 3: Train and Evaluate the Vision Models

### Task 7: Define Training Function

Create a function that:
- Initializes the Cross-Entropy loss function, Adam optimizer, and sets the model to training mode.
- Trains a model for multiple epochs, while keeping track of the training loss and accuracy.

In [8]:
## YOUR SOLUTION HERE ##
def train_model(model, train_loader, num_epochs=10, lr=0.001):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        correct, total = 0, 0
        num_batches = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            loss = loss_fn(logits, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1
            preds = logits.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
        
        avg_loss = epoch_loss / num_batches
        accuracy = correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

### Task 8: Train ResNet18 Model
Train the ResNet18 model for `1` epoch with a learning rate of `0.001`.

In [9]:
## YOUR SOLUTION HERE ##
print("\nTraining ResNet18...")
train_model(model_resnet18, train_loader, num_epochs=1, lr=0.001)


Training ResNet18...
Epoch [1/1], Loss: 0.1103, Accuracy: 0.9663


### Task 9: Train MLP Model
Train the MLP model for `3` epochs with a learning rate of `0.001`.

In [10]:
## YOUR SOLUTION HERE ##
print("Training MLP...")
train_model(model_mlp, train_loader, num_epochs=3, lr=0.001)

Training MLP...
Epoch [1/3], Loss: 0.2255, Accuracy: 0.9319
Epoch [2/3], Loss: 0.1007, Accuracy: 0.9692
Epoch [3/3], Loss: 0.0739, Accuracy: 0.9762


### Task 10: Define Prediction Function
Create a function that builds the prediction loop to generate predictions on the testing set images.

In [11]:
## YOUR SOLUTION HERE ##
def predict(model, dataloader, device="cpu"):
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            preds = logits.argmax(dim=1)
            all_predictions.append(preds.cpu().numpy())
            all_labels.append(batch_y.cpu().numpy())
            
    y_pred = np.concatenate(all_predictions)
    y_true = np.concatenate(all_labels)
    return y_pred, y_true

### Task 11: Generate Predictions
Generate predictions for both models on the test set.

In [12]:
## YOUR SOLUTION HERE ##
print("\nGenerating predictions...")
y_pred_mlp, y_true = predict(model_mlp, test_loader, device=device)
y_pred_resnet18, y_true = predict(model_resnet18, test_loader, device=device)


Generating predictions...


### Task 12: Compare Model Performance
Calculate and display accuracy and classification reports for both models.

In [13]:
## YOUR SOLUTION HERE ##
print("\n=== MLP Performance ===")
accuracy_mlp = accuracy_score(y_true, y_pred_mlp)
print(f"Test Accuracy: {accuracy_mlp:.4f}")
report_mlp = classification_report(y_true, y_pred_mlp, 
                                   target_names=[str(i) for i in range(10)])
print(report_mlp)

print("\n=== ResNet18 Performance ===")
accuracy_resnet18 = accuracy_score(y_true, y_pred_resnet18)
print(f"Test Accuracy: {accuracy_resnet18:.4f}")
report_resnet18 = classification_report(y_true, y_pred_resnet18,
                                       target_names=[str(i) for i in range(10)])
print(report_resnet18)


=== MLP Performance ===
Test Accuracy: 0.9707
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       980
           1       0.97      0.99      0.98      1135
           2       0.95      0.98      0.97      1032
           3       0.96      0.98      0.97      1010
           4       0.98      0.96      0.97       982
           5       0.98      0.96      0.97       892
           6       0.97      0.98      0.98       958
           7       0.96      0.97      0.97      1028
           8       0.98      0.95      0.97       974
           9       0.97      0.96      0.96      1009

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000


=== ResNet18 Performance ===
Test Accuracy: 0.9846
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       1.00      0.99

## Conclusion (Part A)

Compare the results of the MLP and ResNet18 models:
- Which model performed better?
- What are the advantages of CNNs over MLPs for image classification?
- How do the training times compare?
- Which digits were most difficult to classify for each model?

### Task (Optional): Clean Up

In [14]:
# Clean Up Part A
import gc, torch

# 1) Move models off GPU (safe no-op on CPU)
for m in [globals().get('model_mlp'), globals().get('model_resnet18')]:
    try:
        m.to('cpu')
    except Exception:
        pass

# 2) Delete unneeded items
for name in [
    'model_mlp', 'optimizer_mlp', 'scheduler_mlp', 'scaler_mlp', 'criterion_mlp',
    'y_pred_mlp', 'y_true',  
     'train_loader', 'test_loader',     
     'train_dataset', 'test_dataset',
]:
    globals().pop(name, None)

# 3) Collect + clear CUDA cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

## Objective B - Predict Hourly Household Power Usage

## Task Group 4: Household Power Data Loading and Preprocessing

**Introduction to the Household Power Dataset**

The [Household Electric Power Consumption dataset](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption) contains real-world one-minute measurements of one household's electric power consumption (in kilowatts) over four years. 

Key features include:
- `Global_active_power`: Total active power consumed (kilowatts)  
- `Global_reactive_power`: Reactive power (kilowatts)  
- `Voltage`: Average voltage (volts)  
- `Global_intensity`: Current intensity (amperes)  
- `Sub_metering_1`, `Sub_metering_2`, `Sub_metering_3`: Energy usage from different sub-circuits (watt-hours)
- `Timestamp`: Represents the date and time of each hourly observation

For this task, we'll use a smaller subset of the full dataset containing **hourly measurements** instead. Predicting hourly consumption instead of one-minute consumption may help reduce noise, shorten sequence lengths, and reduce overfitting while still preserving daily/weekly patterns.

**Note:** If you find that your kernel crashes or restarts, be sure to run the cleanup cell above from the previous task. It may also be useful to restart the notebook in the **Kernel > Restart Kernel** tab and begin coding from here. 


### Setup and Imports

In [15]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Task 13: Load the Training Dataset

Load the training dataset from `"datasets/household_power_train.csv"` into a pandas and set the `"Timestamp"` column as the DataFrame index.

In [16]:
## YOUR SOLUTION HERE ##
power_train_df = pd.read_csv("datasets/household_power_train.csv", index_col="Timestamp")
features = power_train_df.columns

# Show output - preview dataset
power_train_df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Timestamp,,,,,,,
2009-11-26 00:00:00,0.537150,0.163372,244.434000,2.268333,0.0,0.541667,4.008333
2009-11-26 01:00:00,0.315333,0.108294,244.519583,1.283333,0.0,0.000000,0.650000
2009-11-26 02:00:00,0.323370,0.122347,246.978053,1.368333,0.0,0.541667,0.508333
2009-11-26 03:00:00,0.283747,0.064170,243.914364,1.123333,0.0,0.000000,0.616667
2009-11-26 04:00:00,0.322408,0.121933,243.090667,1.354167,0.0,0.591667,0.650000


### Task 14: Create Lagged Features Function
Create a function called `make_lag_df()` that takes a DataFrame, feature columns, and a window size as parameters. The function should create lagged features by shifting each feature column backward in time for the specified window size.

In [17]:
## YOUR SOLUTION HERE ##
def make_lag_df(df, feature_cols, window=24):
    out = {}
    for col in feature_cols:
        s = df[col]
        for lag in range(window, 0, -1):
            out[f"{col}_t-{lag}"] = s.shift(lag)
    df_seq = pd.DataFrame(out, index=df.index).dropna()
    return df_seq

### Task 15: Apply Lagged Features
Apply the `make_lag_df()` function to create the lagged training DataFrame with a lagged window of 24 hours. 

Be sure to save the lagged features as a new list.

In [18]:
## YOUR SOLUTION HERE ##
power_train_seq_df = make_lag_df(power_train_df, features, window=24)
lagged_features = power_train_seq_df.columns
power_train_seq_df.head()

,Global_active_power_t-24,Global_active_power_t-23,Global_active_power_t-22,Global_active_power_t-21,Global_active_power_t-20,Global_active_power_t-19,Global_active_power_t-18,Global_active_power_t-17,Global_active_power_t-16,Global_active_power_t-15,...,Sub_metering_3_t-10,Sub_metering_3_t-9,Sub_metering_3_t-8,Sub_metering_3_t-7,Sub_metering_3_t-6,Sub_metering_3_t-5,Sub_metering_3_t-4,Sub_metering_3_t-3,Sub_metering_3_t-2,Sub_metering_3_t-1
Timestamp,,,,,,,,,,,,,,,,,,,,,
2009-11-27 00:00:00,0.537150,0.315333,0.323370,0.283747,0.322408,0.275073,1.093236,2.113633,2.278817,1.859634,...,0.600000,0.588333,1.420000,0.533333,1.708333,0.683333,10.966667,17.025000,0.754167,2.1375
2009-11-27 01:00:00,0.315333,0.323370,0.283747,0.322408,0.275073,1.093236,2.113633,2.278817,1.859634,1.492059,...,0.588333,1.420000,0.533333,1.708333,0.683333,10.966667,17.025000,0.754167,2.137500,0.7500
2009-11-27 02:00:00,0.323370,0.283747,0.322408,0.275073,1.093236,2.113633,2.278817,1.859634,1.492059,1.416307,...,1.420000,0.533333,1.708333,0.683333,10.966667,17.025000,0.754167,2.137500,0.750000,0.6150
2009-11-27 03:00:00,0.283747,0.322408,0.275073,1.093236,2.113633,2.278817,1.859634,1.492059,1.416307,1.345167,...,0.533333,1.708333,0.683333,10.966667,17.025000,0.754167,2.137500,0.750000,0.615000,1.6550
2009-11-27 04:00:00,0.322408,0.275073,1.093236,2.113633,2.278817,1.859634,1.492059,1.416307,1.345167,0.554033,...,1.708333,0.683333,10.966667,17.025000,0.754167,2.137500,0.750000,0.615000,1.655000,0.8300


### Task 16: Create Target Variable
Create a target column that contains the next hour's `Global_active_power` values. Use the original DataFrame's values aligned by index and convert the target column to float values.

In [19]:
## YOUR SOLUTION HERE ##
power_train_seq_df["Target"] = power_train_df.loc[power_train_seq_df.index, "Global_active_power"].astype("float32")

### Task 17: Create DataLoader Function
Define a function `df_to_loader()` that converts a sequential DataFrame into a PyTorch DataLoader. The function should reshape the flat lagged features into 3D tensors (batch, timesteps, features) and create a TensorDataset.

In [20]:
from torch.utils.data import TensorDataset, DataLoader

## YOUR SOLUTION HERE ##
def df_to_loader(df_seq, feature_cols, lagged_feature_cols, target_col="Target",
                 window=24, batch_size=32, shuffle=False):
    Xflat = torch.tensor(df_seq[lagged_feature_cols].to_numpy(), dtype=torch.float32)
    N = Xflat.shape[0]
    T = window
    F = len(feature_cols)
    X = Xflat.view(N, T, F)
    y = torch.tensor(df_seq[target_col].to_numpy(), dtype=torch.float32).unsqueeze(1)
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return loader

### Task 18: Create Training DataLoader
Create a training DataLoader by calling `df_to_loader()` with the sequential training data. Specify a batch size and be sure to shuffle the training data.

In [21]:
## YOUR SOLUTION HERE ##
train_loader = df_to_loader(power_train_seq_df,
                            feature_cols=features,
                            lagged_feature_cols=lagged_features,
                            target_col="Target",
                            window=24,
                            batch_size=32,
                            shuffle=True)

## Task Group 5: Build and Train Neural Network Models

### Task 19: Define RNN, GRU, and LSTM Models

**A.** Define a class for an RNN that inherits from `nn.Module`. Include an RNN layer with configurable input size, hidden size, and number of layers, followed by a fully connected layer that outputs a single value.

**B.** Define a class for a GRU that inherits from `nn.Module`. Include a GRU layer with the same configuration as SimpleRNN, followed by a fully connected output layer.

**C.** Define a class for an LSTM that inherits from `nn.Module`. Include an LSTM layer with the same configuration, followed by a fully connected output layer.

In [22]:
## YOUR SOLUTION HERE ##
class SimpleRNN(nn.Module):
    def __init__(self, input_size=7, hidden=64, num_layers=2):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size,
                          hidden_size=hidden,
                          num_layers=num_layers,
                          batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

class SimpleGRU(nn.Module):
    def __init__(self, input_size=7, hidden=64, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size,
                          hidden_size=hidden,
                          num_layers=num_layers,
                          batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

class SimpleLSTM(nn.Module):
    def __init__(self, input_size=7, hidden=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden,
                            num_layers=num_layers,
                            batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

### Task 20: Instantiate Models
Instantiate all three model architectures (RNN, GRU, LSTM) with `input_size=7`, `hidden=64`, and `num_layers=2`. Move each model to the appropriate device.

In [23]:
## YOUR SOLUTION HERE ##
rnn = SimpleRNN(input_size=7, hidden=64, num_layers=2).to(device)
gru = SimpleGRU(input_size=7, hidden=64, num_layers=2).to(device)
lstm = SimpleLSTM(input_size=7, hidden=64, num_layers=2).to(device)

### Task 21: Define Loss Function and Optimizers
Define the Mean Squared Error (MSE) loss function. For each model, create Adam optimizers with a learning rate of `0.001`.

In [24]:
import torch.optim as optim

## YOUR SOLUTION HERE ##
loss_fn = nn.MSELoss()
optimizer_rnn = optim.Adam(rnn.parameters(), lr=0.001)
optimizer_gru = optim.Adam(gru.parameters(), lr=0.001)
optimizer_lstm = optim.Adam(lstm.parameters(), lr=0.001)

### Task 22: Create Training Function
Create a reusable training function called `train_model()` that takes a model, optimizer, train_loader, number of epochs, and model name as parameters. The function should train the model and print the loss for each epoch.

In [25]:
## YOUR SOLUTION HERE ##
def train_model(model, optimizer, train_loader, num_epochs=5, model_name="Model"):
    model.train()
    print(f"Training {model_name}...")
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1
        avg_loss = epoch_loss / num_batches
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}')
    print(f"{model_name} training complete!\n")

### Task 23: Train the RNN, GRU, and LSTM Models
Use the `train_model()` function to separately train all three models (RNN, GRU, LSTM) for `10` epochs each.

In [26]:
## YOUR SOLUTION HERE ##
train_model(rnn, optimizer_rnn, train_loader, num_epochs=10, model_name="RNN")
train_model(gru, optimizer_gru, train_loader, num_epochs=10, model_name="GRU")
train_model(lstm, optimizer_lstm, train_loader, num_epochs=10, model_name="LSTM")

Training RNN...
Epoch [1/10], Loss: 0.460333
Epoch [2/10], Loss: 0.369005
Epoch [3/10], Loss: 0.330579
Epoch [4/10], Loss: 0.310448
Epoch [5/10], Loss: 0.299232
Epoch [6/10], Loss: 0.288518
Epoch [7/10], Loss: 0.271133
Epoch [8/10], Loss: 0.263392
Epoch [9/10], Loss: 0.251117
Epoch [10/10], Loss: 0.245022
RNN training complete!

Training GRU...
Epoch [1/10], Loss: 0.520980
Epoch [2/10], Loss: 0.404374
Epoch [3/10], Loss: 0.361067
Epoch [4/10], Loss: 0.320310
Epoch [5/10], Loss: 0.288701
Epoch [6/10], Loss: 0.271124
Epoch [7/10], Loss: 0.257261
Epoch [8/10], Loss: 0.245090
Epoch [9/10], Loss: 0.234272
Epoch [10/10], Loss: 0.218450
GRU training complete!

Training LSTM...
Epoch [1/10], Loss: 0.529730
Epoch [2/10], Loss: 0.419413
Epoch [3/10], Loss: 0.370221
Epoch [4/10], Loss: 0.340102
Epoch [5/10], Loss: 0.323337
Epoch [6/10], Loss: 0.300306
Epoch [7/10], Loss: 0.278312
Epoch [8/10], Loss: 0.266503
Epoch [9/10], Loss: 0.253184
Epoch [10/10], Loss: 0.244466
LSTM training complete!



## Task Group 6: Generate RNN, GRU, and LSTM Predictions

### Task 24: Define Prediction Function
Define a `predict()` function that creates the prediction loop for each RNN. The function should take a model and DataLoader, sets the model to evaluation mode, and generates predictions for all batches. The function should return the predictions across all batches.

In [27]:
## YOUR SOLUTION HERE ##
def predict(model, dataloader):
    model.eval()
    batch_preds = []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            batch_pred = model(X_batch.to(device))
            batch_pred = batch_pred.cpu().detach().numpy().ravel()
            batch_preds.append(batch_pred)
    y_preds = np.concatenate(batch_preds)
    return y_preds

### Task 25: Prepare Test Data
Load the test data from `"datasets/household_power_test.csv"`, create lagged features, add the target column, and make the test DataLoader without shuffling. 

In [28]:
## YOUR SOLUTION HERE ##
power_test_df = pd.read_csv("datasets/household_power_test.csv", index_col="Timestamp")
power_test_seq_df = make_lag_df(power_test_df, features, window=24)
power_test_seq_df["Target"] = power_test_df.loc[power_test_seq_df.index, "Global_active_power"].astype("float32")

test_loader = df_to_loader(power_test_seq_df,
                           feature_cols=features,
                           lagged_feature_cols=lagged_features,
                           target_col="Target",
                           window=24,
                           batch_size=128,
                           shuffle=False)

### Task 26: Generate Model Predictions
Generate predictions for all three models on the test set using the `predict()` function.

In [29]:
## YOUR SOLUTION HERE ##
rnn_preds = predict(rnn, test_loader)
gru_preds = predict(gru, test_loader)
lstm_preds = predict(lstm, test_loader)

### Task 27: Calculate MSE
Calculate the Mean Squared Error (MSE) for each model's predictions and print the results in a formatted comparison.

In [30]:
## YOUR SOLUTION HERE ##
y_true = power_test_seq_df["Target"].values

mse_rnn = mean_squared_error(y_true, rnn_preds)
mse_gru = mean_squared_error(y_true, gru_preds)
mse_lstm = mean_squared_error(y_true, lstm_preds)

print("=== Test Performance ===")
print(f"RNN  MSE: {mse_rnn:.6f}")
print(f"GRU  MSE: {mse_gru:.6f}")
print(f"LSTM MSE: {mse_lstm:.6f}")

=== Test Performance ===
RNN  MSE: 0.349727
GRU  MSE: 0.333970
LSTM MSE: 0.364201


### Task 28: Display Next-Hour Predictions
Extract the second-to-last prediction from each model that predicts the hourly power consumption of the last observation in the test set. 

Display the predictions alongside the hour's actual power consumption of the last observation.

In [31]:
## YOUR SOLUTION HERE ##
# Use the second-to-last row to predict the last row's hourly power.
pred_power_rnn = rnn_preds[-2]
pred_power_gru = gru_preds[-2]
pred_power_lstm = lstm_preds[-2]

last_obs_power = float(power_test_df["Global_active_power"].iloc[-1])

print(f"\nLast Hour Power:  {last_obs_power:.4f} kW")
print(f"RNN  Prediction:      {pred_power_rnn:.4f} kW")
print(f"GRU  Prediction:      {pred_power_gru:.4f} kW")
print(f"LSTM Prediction:      {pred_power_lstm:.4f} kW")


Last Hour Power:  0.9768 kW
RNN  Prediction:      0.7142 kW
GRU  Prediction:      0.8950 kW
LSTM Prediction:      0.9193 kW


### Task 29: Calculate Model Size

Create a function named `model_size_bytes` that returns a PyTorch model's size in bytes (parameters + buffers). Apply the function to obtain the size of the RNN, GRU, and LSTM. 

In [32]:
def model_size_bytes(model):
    """Return model size in bytes (params + buffers)."""
    model_params = list(model.parameters()) + list(model.buffers())
    size = 0
    for t in model_params:
        size += t.numel() * t.element_size()
    return size

rnn_size = model_size_bytes(rnn)
gru_size = model_size_bytes(gru)
lstm_size = model_size_bytes(lstm)

# Show output - Model Size
print(f"[Model Size] {rnn_size:,} bytes ~ {rnn_size / (1024**2):.3f} MB")
print(f"[Model Size] {gru_size:,} bytes ~ {gru_size / (1024**2):.3f} MB")
print(f"[Model Size] {lstm_size:,} bytes ~ {lstm_size / (1024**2):.3f} MB")

[Model Size] 52,228 bytes ~ 0.050 MB
[Model Size] 156,164 bytes ~ 0.149 MB
[Model Size] 208,132 bytes ~ 0.198 MB


## Conclusion (Part B)

Compare the results of the RNN, GRU, and LSTM:
- Which model produced the lowest MSE on the test set? What about the highest?
- Which model predicted the last hourly power usage the closest to the actual value?
- Was there a noticeable relationship between performance and model size?
- How could we adjust the training process to improve the performance? Think about how the training data was preprocessed into sequences.